# Define input variables and set them as enviornmental variables

In [1]:
%%writefile ./PGS_calc_param.txt

PGS_ID = "PGS002308"
BUILD = "GRCh38" #AoU data is on GRCh38 so this is desired
CHROMS = "1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 X" #Most likely want to include all chr unless are debugging
#CHROMS = "18 19"
#CHROMS = "1 2 3 4 5 6 7 8"

# Save Python variables as environment variables so that they are easier to use within %%bash cells and between notebooks
os.environ["PGS_ID"] = str(PGS_ID)
os.environ["BUILD"] = str(BUILD)
os.environ["CHROMS"] = str(CHROMS)

Overwriting ./PGS_calc_param.txt


# Run here with master script to execute all notebooks in the PGS calculation process

In [3]:
%%writefile run_workflow.py

import papermill as pm
from pathlib import Path
from datetime import datetime
import os
import time
import threading

exec(open("./PGS_calc_param.txt").read())

notebooks = [
    "001_prepare_weights_files_from_PGS_catalog.ipynb",
    "005_run_plink_bash.ipynb",
    "010_combine_chr_check_QC.ipynb",
    "015_genetic_ancestry_adjustment.ipynb",
]

nb_dir = Path("/home/jupyter/workspace/workspace-bucket/calculate_pgs")

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_dir = nb_dir / "workflow_runs" / f"{PGS_ID}_{timestamp}"
run_dir.mkdir(parents=True, exist_ok=True)


def mem_available_gb():
    with open("/proc/meminfo") as f:
        for line in f:
            if line.startswith("MemAvailable:"):
                return int(line.split()[1]) / 1024 / 1024

def mem_total_gb():
    with open("/proc/meminfo") as f:
        for line in f:
            if line.startswith("MemTotal:"):
                return int(line.split()[1]) / 1024 / 1024

def format_seconds(seconds):
    seconds = int(seconds)
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    return f"{h:02d}:{m:02d}:{s:02d}"

def monitor_memory(stop_event, stats, interval_seconds=30):
    total_gb = mem_total_gb()
    max_used_gb = 0

    while not stop_event.is_set():
        available_gb = mem_available_gb()
        used_gb = total_gb - available_gb
        max_used_gb = max(max_used_gb, used_gb)

        stats["total_gb"] = total_gb
        stats["max_used_gb"] = max_used_gb
        stats["min_available_gb"] = total_gb - max_used_gb

        time.sleep(interval_seconds)

    # one final sample
    available_gb = mem_available_gb()
    used_gb = total_gb - available_gb
    max_used_gb = max(max_used_gb, used_gb)

    stats["total_gb"] = total_gb
    stats["max_used_gb"] = max_used_gb
    stats["min_available_gb"] = total_gb - max_used_gb

workflow_start = time.time()
summary = []

for nb in notebooks:
    input_nb = nb_dir / nb
    output_nb = run_dir / nb.replace(".ipynb", "_executed.ipynb")

    print(f"Running {input_nb}", flush=True)

    stop_event = threading.Event()
    mem_stats = {}
    monitor = threading.Thread(
        target=monitor_memory,
        args=(stop_event, mem_stats),
        daemon=True,
    )

    notebook_start = time.time()
    monitor.start()

    try:
        pm.execute_notebook(
            input_path=str(input_nb),
            output_path=str(output_nb),
            kernel_name="python3",
        )
    finally:
        stop_event.set()
        monitor.join()

    elapsed = time.time() - notebook_start

    summary.append({
        "notebook": nb,
        "elapsed": elapsed,
        "max_used_gb": mem_stats.get("max_used_gb"),
        "min_available_gb": mem_stats.get("min_available_gb"),
    })

    print(f"Finished {nb}", flush=True)
    print(f"Elapsed: {format_seconds(elapsed)}", flush=True)
    print(f"Max memory used: {mem_stats.get('max_used_gb', float('nan')):.1f} GB", flush=True)
    print(f"Min memory available: {mem_stats.get('min_available_gb', float('nan')):.1f} GB", flush=True)

workflow_elapsed = time.time() - workflow_start

print("\nWorkflow summary", flush=True)
print("----------------", flush=True)

for item in summary:
    print(
        f"{item['notebook']}: "
        f"elapsed={format_seconds(item['elapsed'])}, "
        f"max_memory_used={item['max_used_gb']:.1f} GB, "
        f"min_memory_available={item['min_available_gb']:.1f} GB",
        flush=True,
    )

print(f"\nTotal workflow time: {format_seconds(workflow_elapsed)}", flush=True)
print(f"Executed notebook copies saved in: {run_dir}", flush=True)

Overwriting run_workflow.py


In [4]:
%%bash
cd /home/jupyter/workspace/workspace-bucket/calculate_pgs
nohup python run_workflow.py > workflow.log 2>&1 &
echo $!

6676
